# RETINA-AI — Colab'da GPU ile Eğitim

Bu notebook `retina-ai-diabetic-retinopathy` reposunu klonlar, Kaggle'dan veri setini indirir ve
modeli Colab'ın ücretsiz GPU'suyla eğitir.

**Önce:** Çalışma zamanı türünü GPU yapın: `Çalışma Zamanı > Çalışma zamanı türünü değiştir > GPU`.

In [ ]:
import torch
print('CUDA kullanılabilir mi:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'yok')

## 1) Repoyu klonla

In [ ]:
!git clone https://github.com/yutronax/retina-ai-diabetic-retinopathy.git
%cd retina-ai-diabetic-retinopathy
!pip install -q -r requirements.txt

## 2) Kaggle API anahtarını yükle

`kaggle.json` dosyanızı (Kaggle hesabı > Settings > Create New Token) aşağıdaki hücreyi çalıştırınca açılacak
dosya seçiciden yükleyin.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # kaggle.json seçin
os.makedirs('/root/.kaggle', exist_ok=True)
for fname in uploaded:
    os.rename(fname, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

## 3) Veri setini indir

In [ ]:
!kaggle datasets download -d kushagratandon12/diabetic-retinopathy-balanced -p data/raw --unzip
!find data/raw -maxdepth 4 -type d

## 4) Veri klasör yolunu doğrula

Yukarıdaki çıktıda `train/val/test` klasörlerinin bulunduğu asıl dizini bulup `DATA_DIR` değişkenine yazın
(örn. `data/raw/content/Diabetic_Balanced_Data`).

In [ ]:
DATA_DIR = 'data/raw/content/Diabetic_Balanced_Data'  # gerekirse düzenleyin
import os
assert os.path.isdir(os.path.join(DATA_DIR, 'train')), f'{DATA_DIR}/train bulunamadı, yolu kontrol edin'

## 5) Eğitim

GPU ile `configs/config.yaml` içindeki tam ayarlarla (25 epoch) eğitim yapılır.

In [ ]:
!python -m src.train --data-dir "$DATA_DIR" --config configs/config.yaml

## 6) Değerlendirme (test seti üzerinde)

In [ ]:
!python -m src.evaluate --data-dir "$DATA_DIR" --checkpoint outputs/checkpoints/best_model.pt --config configs/config.yaml

from IPython.display import Image, display
display(Image('outputs/logs/confusion_matrix.png'))

## 7) En iyi modeli bilgisayarınıza indirin

In [ ]:
from google.colab import files
files.download('outputs/checkpoints/best_model.pt')